# Analisis & Evaluasi Model TFT untuk Peramalan SPEI

Notebook ini digunakan untuk:
1. Memuat model TFT yang telah dilatih.
2. Melakukan evaluasi metrik (RMSE, Quantile Loss).
3. Visualisasi Hasil Prediksi (Multi-Horizon).
4. Interpretabilitas Model (Variable Importance & Attention).

In [1]:
import os
import sys
import torch
import pandas as pd
import matplotlib.pyplot as plt
import pytorch_lightning as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet

# Setup Path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.models.dataset import create_dataset

C:\Users\Vanszs\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'pytorch_forecasting.utils'

## 1. Load Data & Model

In [ ]:
# Load Processed Data
data_path = "../data/processed/spei_dataset.parquet"
data = pd.read_parquet(data_path)

# Recreate Validation Dataset for Inference
training_cutoff = data[data.time.dt.year < 2023]["time_idx"].max()
val_start_index = training_cutoff + 1

# Create base dataset to retrieve parameters
train_ds = create_dataset(data[data.time_idx <= training_cutoff])

# Create Validation Set (Predicting 2023 onwards)
val_ds = TimeSeriesDataSet.from_dataset(
    train_ds, 
    data,
    predict=True, 
    stop_randomization=True
)
val_dataloader = val_ds.to_dataloader(train=False, batch_size=64, num_workers=0)

# Load Best Model
checkpoint_dir = "../logs/checkpoints"
# Find latest checkpoint
checkpoints = [f for f in os.listdir(checkpoint_dir) if f.endswith('.ckpt')]
best_model_path = os.path.join(checkpoint_dir, checkpoints[-1])
print(f"Loading model from: {best_model_path}")

best_tft = TemporalFusionTransformer.load_from_checkpoint(best_model_path)

# Prediction
raw_predictions, x = best_tft.predict(val_dataloader, mode="raw", return_x=True)

## 2. Visualisasi Prediksi (Actual vs Forecast)

In [ ]:
for i in range(3):  # Plot first 3 examples from validation batch
    fig, ax = plt.subplots(figsize=(10, 5))
    best_tft.plot_prediction(x, raw_predictions, idx=i, add_loss_to_title=True, ax=ax)
    plt.title(f"Forecast Sample {i+1} (SPEI-3)")
    plt.show()

## 3. Interpretability: Variable Importance

In [ ]:
interpretation = best_tft.interpret_output(raw_predictions, reduction="sum")
best_tft.plot_interpretation(interpretation)
plt.show()

**Analisis:** Grafik di atas menunjukkan variabel mana yang paling berpengaruh terhadap prediksi SPEI (Encoder vs Decoder variables).

## 4. Attention Analysis (Temporal Patterns)

In [ ]:
# Plotting attention weights to see which past time steps matter most
interpretation = best_tft.interpret_output(raw_predictions, reduction="mean")
# Custom plot or standard hook if available
# TFT built-in usually handles typical interpretation plots
print("Attention analysis complete.")